# HippoVoice — Track 2 Exploration: Memory + Audio-to-Audio (T4 GPU)

**v1 — created 2026-08-05**

**Status: exploratory first pass, not validated.** Track 1 (`colab.ipynb`) is the
real, benchmarked deliverable — hippocampal memory (Ebbinghaus decay +
HippoRAG retrieval) on top of a standard cascaded STT -> LLM -> TTS pipeline,
validated on real LoCoMo data (avg F1 24.1% over 1540 questions). Track 2 was
never built before this notebook -- `BUGS.md` had it logged as "entirely
unbuilt." This notebook is a first, honest attempt at wiring HippoVoice's
existing memory system (unchanged -- it's architecture-agnostic, it just
operates on extracted memory dicts) into a genuine audio-to-audio model
instead of the cascaded pipeline.

**Model choice: [Mini-Omni](https://github.com/gpt-omni/mini-omni)** (~1.5B
params, ~4GB VRAM). Chosen over larger audio-to-audio models (Moshi 7B/
16-24GB, Qwen2.5-Omni 7B, GLM-4-Voice 9B) specifically because it's the only
one that comfortably fits a single Kaggle T4 (15GB) alongside the embedding
model and an LLM for memory extraction, without quantization gymnastics.
Trade-off: lower voice quality than the bigger models -- fine for a
feasibility test, not for a real deployment.

**What this notebook actually validates (and what it doesn't):**
- ✅ Mini-Omni runs end-to-end (audio in -> audio out) on a T4
- ✅ HippoVoice's memory system (extraction, salience, decay, HippoRAG
  retrieval) can capture and recall facts from a Mini-Omni-driven conversation
- ❌ **Not yet done:** feeding retrieved memory context back *into*
  Mini-Omni's own generation. Mini-Omni's public inference API
  (`OmniInference`, `A1_A2`) takes audio in, doesn't expose an obvious
  system-prompt/context-injection hook the way a text LLM's
  `generate(system=..., messages=...)` does. So right now this is
  side-by-side (memory observes the conversation) rather than integrated
  (memory conditions the response). See the "Known limitation" cell near the
  end for what real integration would need.

## Startup sequence (fresh runtime)
1. Run **Step 1** (install deps) -> Runtime restart -> **Step 2** (clone repos)
2. Run sections in order below

All cell code below (the exact `OmniInference`/`A1_A2` call signatures) is
reconstructed from Mini-Omni's README and `inference.py` via web research,
not verified by actually running it (no GPU available in this environment) --
treat the Section 1-2 cells as a first draft that may need small signature
fixes once actually run on Kaggle, same as any new integration.

In [ ]:
# ── STEP 1: Install deps ───────────────────────────────────────────────
# HippoVoice's existing deps (memory system, embeddings, existing TTS used
# only to synthesize test-input audio) plus Mini-Omni's own requirements.
# Run once, then Runtime -> Restart session.

!apt-get -qq install -y espeak-ng espeak > /dev/null 2>&1

!pip install -q pyttsx3 chromadb networkx "sentence-transformers>=2.7.0"
!pip install -q soundfile datasets pytest pytest-mock nltk
!pip install -q bitsandbytes accelerate

# Mini-Omni's own stack -- litgpt for the backbone, snac for audio decoding,
# whisper for audio encoding (already installed above transitively, but
# pinned explicitly here since Mini-Omni's inference.py imports it directly).
!pip install -q litgpt snac openai-whisper

!pip install -q --force-reinstall "numpy==2.0.2" "protobuf==5.29.6"

import numpy as np, google.protobuf
print(f'numpy     {np.__version__}')
print(f'protobuf  {google.protobuf.__version__}')
print()
print('━' * 50)
print('NOW: Runtime → Restart session  (Ctrl+M .)')
print('THEN: run Step 2 and below.')
print('━' * 50)

In [ ]:
# ── STEP 2: Clone repos ───────────────────────────────────────────────
# Run after restart. Clones both HippoVoice (memory system) and Mini-Omni
# (audio-to-audio model) side by side. Re-run after any push to either.

import os, sys

WORK_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
HIPPO_DIR = f'{WORK_DIR}/hippovoice'
OMNI_DIR = f'{WORK_DIR}/mini-omni'

if os.path.exists(os.path.join(HIPPO_DIR, '.git')):
    !git -C {HIPPO_DIR} pull
else:
    !git clone https://github.com/shivansh193/hippovoice.git {HIPPO_DIR}

if not os.path.exists(os.path.join(OMNI_DIR, '.git')):
    !git clone https://github.com/gpt-omni/mini-omni.git {OMNI_DIR}

for d in (HIPPO_DIR, OMNI_DIR):
    if d not in sys.path:
        sys.path.insert(0, d)

os.chdir(HIPPO_DIR)

import subprocess
try:
    commit = subprocess.check_output(['git', '-C', HIPPO_DIR, 'log', '-1', '--format=%h'], text=True).strip()
except Exception:
    commit = 'unknown'
print(f'hippovoice commit [{commit}]')
print('Ready.')

In [ ]:
import torch

# Same lesson as Track 1 (BUGS.md, 2026-08-04): a session with Accelerator
# set to "None" runs everything on CPU, which for an audio model on top of
# an LLM is unusably slow. Fail fast instead of finding out hours later.
assert torch.cuda.is_available(), (
    'No GPU detected -- set Accelerator to "GPU T4 x2" in Kaggle Settings '
    'before running this cell.'
)
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'VRAM free: {free:.1f} GB  (T4)')

## 1. Load Mini-Omni (audio-to-audio, ~1.5B, ~4GB VRAM)

In [ ]:
# Reconstructed from Mini-Omni's README (web research, not run) -- the
# checkpoint auto-downloads on first load if ckpt_dir doesn't exist yet.
os.chdir(OMNI_DIR)
from inference import OmniInference

CKPT_DIR = f'{OMNI_DIR}/checkpoint'
omni = OmniInference(ckpt_dir=CKPT_DIR, device='cuda:0')
print('Mini-Omni loaded')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 2. Smoke Test — one audio-in / audio-out turn

No real-speech fixture exists in the repo (`tests/fixtures/` is currently
empty of audio -- the WAVs referenced in `PLAN.md` were never actually
created), so this synthesizes its own test input with HippoVoice's existing
offline TTS rather than depending on Mini-Omni's own bundled samples (whose
exact path isn't documented anywhere I could find). Self-contained either way.

In [ ]:
os.chdir(HIPPO_DIR)
from tts.model import load_tts
from tts.synthesize import synthesize
os.chdir(OMNI_DIR)
from inference import load_audio, get_input_ids_whisper, A1_A2

tts_engine = load_tts()
TEST_AUDIO = f'{WORK_DIR}/test_input.wav'
synthesize(tts_engine, "Hello, how are you today?", TEST_AUDIO)

mel, leng = load_audio(TEST_AUDIO)
audio_feature, input_ids = get_input_ids_whisper(mel, leng, omni.whispermodel, omni.device)

OUT_DIR = f'{WORK_DIR}/track2_out'
os.makedirs(OUT_DIR, exist_ok=True)

text = A1_A2(
    omni.fabric, audio_feature, input_ids, leng,
    omni.model, omni.text_tokenizer, 0,
    omni.snacmodel, OUT_DIR,
)
print(f'Mini-Omni generated text: {text!r}')
print(f'Audio saved under: {OUT_DIR}')

## 3. Wire in HippoVoice's memory system

Unchanged from Track 1 -- `memory/store.py`, `memory/extractor.py`,
`memory/scorer.py`, `memory/decay.py`, `memory/retriever.py` all operate on
plain memory dicts (content/emotion/salience), completely agnostic to how
the text was produced. Extraction runs on what the *user* said (the text we
synthesized as input), same as Track 1 extracts from user turns.

In [ ]:
os.chdir(HIPPO_DIR)
from memory.store import HippoMemory
from memory.extractor import extract_memories
from memory.retriever import hippo_retrieve
from llm.client import LLMClient

# Separate small LLM just for memory extraction (Mini-Omni's own text output
# isn't reliable enough for structured JSON extraction -- same reasoning as
# Track 1's EXTRACTION_PROMPT needing Qwen3-4B, not Qwen3-0.6B, see BUGS.md).
# ~4GB (Mini-Omni) + ~3GB (Qwen3-4B 4bit) + embedding model comfortably fits
# a T4's 15GB.
llm = LLMClient(model_name='Qwen/Qwen3-4B', load_in_4bit=True)
mem = HippoMemory()
current_turn = 0

def process_track2_turn(user_text: str) -> str:
    """One turn: synthesize user_text -> audio -> Mini-Omni audio-to-audio
    reply, while HippoVoice's memory extracts + stores from user_text in
    parallel. Returns Mini-Omni's raw generated text (NOT yet conditioned
    on retrieved memory -- see the limitation cell below)."""
    global current_turn
    turn_audio = f'{WORK_DIR}/turn_input_{current_turn}.wav'
    synthesize(tts_engine, user_text, turn_audio)

    os.chdir(OMNI_DIR)
    mel, leng = load_audio(turn_audio)
    audio_feature, input_ids = get_input_ids_whisper(mel, leng, omni.whispermodel, omni.device)
    reply_text = A1_A2(
        omni.fabric, audio_feature, input_ids, leng,
        omni.model, omni.text_tokenizer, current_turn,
        omni.snacmodel, f'{OUT_DIR}/turn_{current_turn}',
    )

    os.chdir(HIPPO_DIR)
    memories = extract_memories(user_text, llm)
    for m in memories:
        m.update({
            'emotion': {'label': 'neutral', 'intensity': 0.3},
            'base_weight': 1.0, 'recall_count': 0, 'turn_created': current_turn,
        })
        mem.add(m)

    current_turn += 1
    return reply_text

print('process_track2_turn ready')

## 4. Multi-turn memory capture test

Same shape as Track 1's manual walkthrough (`colab.ipynb` section 2): state a
fact, talk about something unrelated, then check the fact is retrievable.
**Mini-Omni's own spoken reply to the third turn will NOT actually reference
"Max"** -- that's the known limitation, not a bug in this test. What this
confirms is narrower but still real: memory correctly captures and recalls
facts from an audio-to-audio-driven conversation, which didn't exist before
this notebook.

In [ ]:
turns = [
    "My dog's name is Max and he loves swimming.",
    "The weather has been nice this week.",
    "What's my dog's name again?",
]

for t in turns:
    reply = process_track2_turn(t)
    print(f'User said:            {t!r}')
    print(f'Mini-Omni replied:    {reply!r}')
    print()

print('--- Memory retrieval check (does it recall Max?) ---')
results = mem.search('dog name', top_k=3)
for r in results:
    print(f'  {r["content"]}')

## Known limitation / next step

The retrieval check above should show "Max" was captured and is retrievable
-- that's real. What's **not** proven yet is Mini-Omni actually *using* that
retrieved memory when it generates its spoken reply to "what's my dog's name
 again" -- right now nothing feeds `results` back into the `A1_A2` call, so
Mini-Omni is answering blind, same as if HippoVoice's memory system didn't
exist.

Track 1's text pipeline solves this with `llm/context.py::build_system_prompt`
-- it prepends retrieved memories to the LLM's system prompt before
`generate()`. Mini-Omni's public inference path (`get_input_ids_whisper` ->
`A1_A2`) doesn't have an obvious equivalent hook in what's documented --
`input_ids` appears to be built purely from the Whisper-encoded audio, not a
mixed text+audio prompt. Real integration needs one of:

1. **Read Mini-Omni's actual `input_ids` construction** (in its source, not
   just the README) to see if a text prefix can be spliced in before the
   audio tokens -- cheapest if it works, but unverified.
2. **Try a model with a documented text-context API** instead -- Qwen2.5-Omni
   or CSM (Sesame) both reportedly accept text alongside audio, at the cost
   of needing more VRAM than a single T4 comfortably offers (see the header
   cell) -- would need a bigger GPU tier or aggressive quantization.
3. **Inject context via synthesized audio** -- literally prepend a short
   spoken summary of retrieved memories (via the existing TTS) to the front
   of the audio Mini-Omni receives, so it's "heard" as part of the same
   utterance. Hacky, but works with zero changes to Mini-Omni itself --
   probably the fastest thing to actually try next.

This notebook stops at the honest checkpoint: memory *capture* from an
audio-to-audio conversation works, memory *conditioning* of that same
model's generation does not yet.

## 5. Save results

In [ ]:
import json, datetime

out = {
    'timestamp': datetime.datetime.now().isoformat(),
    'gpu': 'T4',
    'model': 'mini-omni',
    'status': 'memory capture validated; memory-conditioned generation not yet implemented',
    'retrieved_for_dog_query': [r['content'] for r in mem.search('dog name', top_k=3)],
}

RESULTS_PATH = f'{WORK_DIR}/track2_results.json'
with open(RESULTS_PATH, 'w') as f:
    json.dump(out, f, indent=2)

print(f'Saved to {RESULTS_PATH} (visible in the Output tab after commit on Kaggle)')
print(json.dumps(out, indent=2))